# 02 · BM25 baseline (Prompt 5, owner: M1)

Before running:
1. **Accelerator: None** (CPU only, saves GPU quota). **Internet: On**.
2. **Add Input → Your Work → Datasets → `fever-corpus-v1`** (the Phase 1 output).

Expected run time: about 20-40 minutes (install, Lucene index of 308k sentences, 12 k1/b settings on
2k validation claims, 20k test claims, 30k train claims).

When it finishes: **Save Version → Save & Run All (Commit)**. Notebook `03_dense` adds this notebook's
output as an input (it needs `outputs/bm25/run_test.jsonl`).

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

## Install Java 21 and Pyserini
Pyserini runs Lucene inside a Java virtual machine and needs Java 21. We try `apt-get` first and fall back to downloading a JDK with `install-jdk`. The JDK goes to `/tmp`, so it is not saved in the notebook output.

In [ ]:
import os, subprocess, sys, glob

def java_home_from_apt():
    cands = sorted(glob.glob("/usr/lib/jvm/java-21-openjdk*"))
    return cands[0] if cands else None

home = java_home_from_apt()
if home is None:
    r = subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-21-jdk-headless", shell=True)
    home = java_home_from_apt()
if home is None:   # fallback: download a JDK 21 into /tmp
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "install-jdk"], check=True)
    import jdk
    home = jdk.install("21", path="/tmp/jdk")
os.environ["JAVA_HOME"] = home
os.environ["PATH"] = os.path.join(home, "bin") + os.pathsep + os.environ["PATH"]
print("JAVA_HOME =", home)
!java -version

In [ ]:
!pip install -q pyserini rank_bm25 pyyaml pytest
!python -c "from pyserini.search.lucene import LuceneSearcher; print('pyserini OK')"

## Unit tests

In [ ]:
!python -m pytest -q -rs

## Quick test: rank_bm25 on a 50k-sentence sample (no Java involved)
Checks the data and evaluation path in about a minute before the full run.

In [ ]:
!python -m src.retrieval.bm25 --config configs/bm25.yaml --quick

## Full run: Pyserini index, k1/b tuning on validation, test metrics

In [ ]:
!python -m src.retrieval.bm25 --config configs/bm25.yaml

In [ ]:
import json, pandas as pd
m = json.load(open("outputs/bm25/metrics.json"))
print(f"best k1 = {m['best_k1']}, b = {m['best_b']} | test split {m['test_split']} | "
      f"{m['ms_per_query']} ms/query (single), {m['batch_ms_per_query']} ms/query (batch)")
print(pd.DataFrame({lv: m["test"][lv] for lv in ("page", "sentence")}).round(4))
pd.DataFrame(m["tuning_grid"])[["k1", "b", "sentence_recall@10", "page_recall@10", "sentence_mrr"]].round(4)

In [ ]:
!du -sh outputs/bm25/*